# CabinGuard-ADI: training and evaluation on Google Colab

Runs every experiment behind the paper's numbers and returns a zip with the new
`models/`, `results/` and `paper_numbers.tex`.

| Step | Data | Needs | Time on a free Colab CPU |
|---|---|---|---|
| 1-3 | code, packages, PhysioNet downloads (~1.1 GB) | nothing | ~5 min |
| 4-6 | SeizeIT2 seizure minutes (OpenNeuro), cardiac and motion branches (UEA Epilepsy, UCI MHEALTH and UCI HAR, ~140 MB), fused evaluation, fault matrix | nothing | ~60-90 min |
| 7-8 | the earlier injected-jerk motion branch on the same external data, for comparison (optional) | nothing | ~5 min |
| 9 | PhysDrive in-car heart rate replay (optional) | Kaggle API token, large download | depends on Kaggle |
| 10-11 | tests, paper numbers, download the outputs | nothing | ~1 min |

A GPU does not help: every model here is scikit-learn on CPU. Use
**Runtime > Change runtime type > CPU** (the high-RAM option is not needed).

## Settings
`GITHUB_TOKEN` is only needed if the repository is private: add it under the key
icon (Colab *Secrets*) with notebook access switched on. For PhysDrive add your Kaggle
API token the same way, as a secret named `KAGGLE_API_TOKEN` (Kaggle > Settings > API >
Generate New Token). The Kaggle part of PhysDrive is MIT-licensed; nothing has to be accepted.

In [ ]:
REPO = "baha-eddine-hammou/Cabin_Guard_TSYP14"
BRANCH = "claude/zen-carson-esouiy"
RUN_CARDIAC = False          # step 5: retrain the cardiac branch (unchanged; its model is already committed)
RUN_EPILEPSY_CHECK = False   # step 7: earlier injected-jerk motion branch on UEA Epilepsy (comparison only)
RUN_UCI_CHECKS = False       # step 8: earlier injected-jerk branch on MHEALTH and HAR (comparison only)
RUN_PHYSDRIVE = False        # step 9: needs Kaggle credentials and a large download
FAULT_SEEDS = 10

## 1. Get the code

In [ ]:
import os, subprocess, pathlib
os.chdir("/content")
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if not pathlib.Path("Cabin_Guard_TSYP14").exists():
    token = secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "Cabin_Guard_TSYP14"])
    if r.returncode != 0:
        print("Clone failed (private repository without a token?). Upload a zip of the repository instead:")
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        subprocess.run(["unzip", "-q", name, "-d", "/content"], check=True)
        cand = [p for p in pathlib.Path("/content").glob("*") if (p / "prototype" / "cabinguard").exists()]
        cand[0].rename("/content/Cabin_Guard_TSYP14")
os.chdir("/content/Cabin_Guard_TSYP14/prototype")
print(os.getcwd())

## 2. Install

In [ ]:
!pip install -q -e ".[dev]"
import os; print("CPU cores:", os.cpu_count())

## 3. Download the PhysioNet recordings
AWS open-data mirror first (fast), physionet.org as fallback, every file checked
against the release's SHA-256 sums. One progress line per file; the last database
(walk-climb-drive accelerometry, ~760 MB) is most of the time.

In [ ]:
!python -m cabinguard.physionet_fetch drivedb szdb vfdb cudb mitdb accelerometry-walk-climb-drive
!du -sh ../data/*

## 4. Extract labelled windows
Includes the neck accelerometry around every annotated seizure of SeizeIT2
(OpenNeuro ds005873, 125 patients): only those minutes are downloaded, by HTTP range
requests, not the 27 GB of motion files.

In [ ]:
import os
W = os.cpu_count()
!python experiments/realdata/extract_cardiac.py --workers {W}
!python experiments/realdata/extract_motion.py --workers {W}
!python experiments/realdata/extract_motion.py --branch --workers {W}
!python experiments/realdata/extract_seizeit2.py --workers 16   # SeizeIT2 clinical seizures (OpenNeuro), ~30 min

## 5. Train and evaluate the branches (held-out records, subjects and patients)
The cardiac branch, then the motion branch trained on recorded motion only: SeizeIT2
motor seizures and UEA Epilepsy seizure mimics against driving, patient background,
everyday activities (UCI MHEALTH) and the mimics' own walking, running and sawing.

In [ ]:
if RUN_CARDIAC:
    !python experiments/realdata/train_branches.py > /tmp/branches.log 2>&1; tail -c 1500 /tmp/branches.log
!python experiments/realdata/train_motion_real.py

## 6. Fused evaluation and the fault matrix

In [ ]:
!python experiments/realdata/evaluate_fusion.py 2>&1 | grep -v "  episode"
!python experiments/fault_matrix.py --seeds {FAULT_SEEDS}

## 7. Earlier injected-jerk motion branch on UEA Epilepsy - optional
The motion branch trained on driving with injected jerks, kept for comparison:
(a) applied unchanged to the recorded seizure mimics and (b) the same features
trained on the official TRAIN split.

In [ ]:
if RUN_EPILEPSY_CHECK:
    !python experiments/realdata/external_checks.py epilepsy

## 8. Earlier injected-jerk branch on everyday activities (UCI MHEALTH, UCI HAR) - optional
How often the injected-jerk branch fires on rhythmic non-driving movement
(jogging, cycling, stairs), for comparison with step 5.

In [ ]:
if RUN_UCI_CHECKS:
    !python experiments/realdata/external_checks.py mhealth har

## 9. PhysDrive in-car heart rate - optional
Kaggle `xiaoyang274/physdrive` (Wang et al., NeurIPS 2025). Replays the real
in-vehicle heart-rate labels through the trained cardiac branch to see how often it
alone would raise an alarm in a moving car. The archive is large; check your Colab
disk first.

In [ ]:
if RUN_PHYSDRIVE:
    import os, sys, pathlib
    !pip install -q -U kagglehub
    token = secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token          # new single-token format
    else:                                               # legacy kaggle.json pair, if you have one
        os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME") or ""
        os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY") or ""
    import kagglehub
    ds = pathlib.Path(kagglehub.dataset_download("xiaoyang274/physdrive"))
    print("PhysDrive downloaded to", ds)
    !du -sh "{ds}"
    sys.path.insert(0, "experiments/colab")
    import colab_physdrive_replay as pd
    pd.ROOT = pathlib.Path("results")
    pd.run_replay(pathlib.Path("."), ds)

## 10. Tests and paper numbers

In [ ]:
!python -m pytest -q
!python experiments/export_paper_numbers.py

## 11. Download the outputs
Unzip into the repository root on your PC (it contains `prototype/models`,
`prototype/results` and `paper_numbers.tex`), then rebuild the paper.

In [ ]:
import shutil, pathlib, os
os.chdir("/content/Cabin_Guard_TSYP14")
out = pathlib.Path("/content/cabinguard_outputs")
if out.exists(): shutil.rmtree(out)
for p in ["prototype/models", "prototype/results", "paper_numbers.tex", "paper_numbers_external.tex"]:
    src = pathlib.Path(p); dst = out / p
    dst.parent.mkdir(parents=True, exist_ok=True)
    (shutil.copytree if src.is_dir() else shutil.copy)(src, dst)
shutil.make_archive("/content/cabinguard_outputs", "zip", out)
from google.colab import files
files.download("/content/cabinguard_outputs.zip")